In [4]:
%pip install scikit-learn pandas numpy matplotlib seaborn joblib streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 97.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 116.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 797.2/797.2 kB 55.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.0/55.0 MB 145.9 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 148.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 159.4 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 112.6 MB/s eta 0:00:00
  Attempting uninstall: typing-extensions
    Found existing installation: typing_extensions 4.11.0
    Uninstalling typing_extensions-4.11.0:
      Successfully uninstalled typing_extensions-4.11.0
  Attempting uninstall: protobuf
    Found existing installation: protobuf 4.25.3
    Uninstalling protobuf-4.25.3:
      Successfully uninstalled protobuf-4.25.3
  Attempting uninstall: numpy
    Found existing installation: num

In [2]:
import sklearn
import pandas
import numpy

print("Scikit-learn:", sklearn.__version__)
print("Pandas:", pandas.__version__)
print("NumPy:", numpy.__version__)
print("All packages installed successfully!")

Scikit-learn: 1.9.1
Pandas: 2.2.3
NumPy: 2.5.4
All packages installed successfully!


In [4]:

# ==========================================
# BANK MARKETING CLASSIFICATION PROJECT
# COMPLETE MODEL TRAINING
# ==========================================

import os
import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import LinearSVC

from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    confusion_matrix,
    classification_report
)

# 1. Load dataset
df = pd.read_csv("bank-additional-full.csv", sep=";")

# 2. Feature selection
# Exclude duration to prevent target leakage
X = df.drop(columns=["y", "duration"])
y = df["y"]

# 3. Identify numerical and categorical features
numerical_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object"]
).columns.tolist()

print("Numerical features:", len(numerical_features))
print("Categorical features:", len(categorical_features))

# 4. Split dataset into training and testing
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\nTraining records:", X_train.shape[0])
print("Testing records:", X_test.shape[0])

# 5. Define preprocessing
def create_preprocessor():
    return ColumnTransformer(
        transformers=[
            ("num", StandardScaler(), numerical_features),
            ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
        ],
        sparse_threshold=0
    )

# 6. Define five classification models
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000, random_state=42
    ),

    "Decision Tree": DecisionTreeClassifier(
        random_state=42
    ),

    "KNN": KNeighborsClassifier(
        n_neighbors=5
    ),

    "Naive Bayes": GaussianNB(),

    "Support Vector Machine": LinearSVC(
        max_iter=5000, random_state=42
    )
}

# 7. Create model folder
os.makedirs("model", exist_ok=True)

results = []
trained_models = {}

# 8. Train and evaluate each model
for name, classifier in models.items():

    print("\nTraining:", name)

    pipeline = Pipeline([
        ("preprocessor", create_preprocessor()),
        ("classifier", classifier)
    ])

    pipeline.fit(X_train, y_train)

    predictions = pipeline.predict(X_test)

    classes = list(pipeline.named_steps["classifier"].classes_)

    if hasattr(pipeline, "predict_proba"):
        scores = pipeline.predict_proba(X_test)[:, classes.index("yes")]
    else:
        scores = pipeline.decision_function(X_test)
        if classes.index("yes") == 0:
            scores = -scores

    accuracy = accuracy_score(y_test, predictions)
    auc = roc_auc_score(y_test, scores)
    precision = precision_score(
        y_test, predictions, pos_label="yes", zero_division=0
    )
    recall = recall_score(
        y_test, predictions, pos_label="yes", zero_division=0
    )
    f1 = f1_score(
        y_test, predictions, pos_label="yes", zero_division=0
    )
    mcc = matthews_corrcoef(y_test, predictions)

    results.append({
        "Model": name,
        "Accuracy": accuracy,
        "AUC": auc,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "MCC": mcc
    })

    # Save trained pipeline
    filename = name.lower().replace(" ", "_") + ".joblib"

    joblib.dump(pipeline, os.path.join("model", filename))

    trained_models[name] = pipeline

    print("Accuracy:", round(accuracy, 4))
    print("AUC:", round(auc, 4))
    print("Precision:", round(precision, 4))
    print("Recall:", round(recall, 4))
    print("F1 Score:", round(f1, 4))
    print("MCC:", round(mcc, 4))

# 9. Create model comparison table
results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by="F1 Score", ascending=False
)

print("\n========== MODEL COMPARISON ==========")
display(results_df.round(4))

# 10. Save comparison results
results_df.to_csv("model_comparison.csv", index=False)

# 11. Save test data with target column
test_data = X_test.copy()
test_data["y"] = y_test

test_data.to_csv("test_data.csv", index=False)

# 12. Display confusion matrices
for name, pipeline in trained_models.items():

    predictions = pipeline.predict(X_test)

    print("\nConfusion Matrix:", name)
    print(confusion_matrix(
        y_test, predictions, labels=["no", "yes"]
    ))

    print("\nClassification Report:", name)
    print(classification_report(
        y_test, predictions, labels=["no", "yes"],
        zero_division=0
    ))

# 13. Verify saved files
print("\n========== SAVED MODEL FILES ==========")

for file in sorted(os.listdir("model")):
    print(file)

print("\nTest data saved:", os.path.exists("test_data.csv"))
print("Comparison saved:", os.path.exists("model_comparison.csv"))

print("\nALL FIVE MODELS TRAINED SUCCESSFULLY!")


Numerical features: 9
Categorical features: 10

Training records: 32950
Testing records: 8238

Training: Logistic Regression
Accuracy: 0.9009
AUC: 0.8008
Precision: 0.6905
Recall: 0.2188
F1 Score: 0.3322
MCC: 0.3516

Training: Decision Tree
Accuracy: 0.8418
AUC: 0.6258
Precision: 0.3131
Recall: 0.3384
F1 Score: 0.3252
MCC: 0.236

Training: KNN
Accuracy: 0.8972
AUC: 0.7444
Precision: 0.5839
Recall: 0.3039
F1 Score: 0.3997
MCC: 0.3719

Training: Naive Bayes
Accuracy: 0.8049
AUC: 0.7755
Precision: 0.3172
Recall: 0.6347
F1 Score: 0.423
MCC: 0.349

Training: Support Vector Machine
Accuracy: 0.9014
AUC: 0.7979
Precision: 0.7164
Recall: 0.2069
F1 Score: 0.3211
MCC: 0.3502

========== MODEL COMPARISON ==========


,Model,Accuracy,AUC,Precision,Recall,F1 Score,MCC
3,Naive Bayes,0.8049,0.7755,0.3172,0.6347,0.4230,0.3490
2,KNN,0.8972,0.7444,0.5839,0.3039,0.3997,0.3719
0,Logistic Regression,0.9009,0.8008,0.6905,0.2188,0.3322,0.3516
1,Decision Tree,0.8418,0.6258,0.3131,0.3384,0.3252,0.2360
4,Support Vector Machine,0.9014,0.7979,0.7164,0.2069,0.3211,0.3502



Confusion Matrix: Logistic Regression
[[7219   91]
 [ 725  203]]

Classification Report: Logistic Regression
              precision    recall  f1-score   support

          no       0.91      0.99      0.95      7310
         yes       0.69      0.22      0.33       928

    accuracy                           0.90      8238
   macro avg       0.80      0.60      0.64      8238
weighted avg       0.88      0.90      0.88      8238


Confusion Matrix: Decision Tree
[[6621  689]
 [ 614  314]]

Classification Report: Decision Tree
              precision    recall  f1-score   support

          no       0.92      0.91      0.91      7310
         yes       0.31      0.34      0.33       928

    accuracy                           0.84      8238
   macro avg       0.61      0.62      0.62      8238
weighted avg       0.85      0.84      0.84      8238


Confusion Matrix: KNN
[[7109  201]
 [ 646  282]]

Classification Report: KNN
              precision    recall  f1-score   support

     

In [6]:
import os

print("Project files:")
print(os.listdir())

print("\nSaved models:")
print(os.listdir("model"))

Project files:
['.mozilla', '.bash_logout', '.bash_profile', '.bashrc', '.config', '.cache', 'Desktop', 'Downloads', 'Templates', 'Public', 'Documents', 'Music', 'Pictures', 'Videos', '.local', '.ICEauthority', '.gnupg', '.vnc', '.Xauthority', 'thinclient_drives', '.xsession-errors.old', '.xsession-errors', '.ssh', 'anaconda3', '.conda', '.ipython', '.vscode', '.pki', '.jupyter', '.ipynb_checkpoints', '.keras', '.anaconda', '.wget-hsts', '.continuum', '.condarc', 'anaconda_projects', '.python_history', '.java', '.viminfo', '.idea', '.bash_history', '.xsession', 'workspace', 'Untitled.ipynb', 'test_data.csv', 'bank-additional-full.csv', 'Untitled1.ipynb', 'model', 'model_comparison.csv']

Saved models:
['logistic_regression.joblib', 'decision_tree.joblib', 'knn.joblib', 'naive_bayes.joblib', 'support_vector_machine.joblib']


In [8]:

%%writefile app.py

import streamlit as st
import pandas as pd
import joblib
import os

from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    confusion_matrix,
    classification_report
)

st.set_page_config(
    page_title="Bank Marketing Classification",
    page_icon="📊",
    layout="wide"
)

st.title("Bank Marketing Classification")
st.write("Evaluate five trained machine learning classification models.")

# Available models
MODEL_FILES = {
    "Logistic Regression": "logistic_regression.joblib",
    "Decision Tree": "decision_tree.joblib",
    "KNN": "knn.joblib",
    "Naive Bayes": "naive_bayes.joblib",
    "Support Vector Machine": "support_vector_machine.joblib"
}

# Input features
FEATURES = [
    "age", "job", "marital", "education", "default",
    "housing", "loan", "contact", "month", "day_of_week",
    "campaign", "pdays", "previous", "poutcome",
    "emp.var.rate", "cons.price.idx", "cons.conf.idx",
    "euribor3m", "nr.employed"
]

# Model selection
model_name = st.selectbox(
    "Select Classification Model",
    list(MODEL_FILES.keys())
)

# CSV upload
uploaded_file = st.file_uploader(
    "Upload Test CSV",
    type=["csv"]
)

if uploaded_file is not None:

    data = pd.read_csv(uploaded_file)

    missing = [
        col for col in FEATURES + ["y"]
        if col not in data.columns
    ]

    if missing:
        st.error(f"Missing columns: {missing}")

    elif data.empty:
        st.error("The uploaded CSV contains no records.")

    else:
        X_test = data[FEATURES]
        y_test = data["y"].astype(str).str.lower().str.strip()

        if not y_test.isin(["yes", "no"]).all():
            st.error("Target column must contain only yes and no.")

        else:
            model_path = os.path.join(
                "model", MODEL_FILES[model_name]
            )

            if not os.path.exists(model_path):
                st.error("Selected model file not found.")

            else:
                model = joblib.load(model_path)

                predictions = model.predict(X_test)

                classifier = model.named_steps["classifier"]
                classes = list(classifier.classes_)

                if hasattr(model, "predict_proba"):
                    scores = model.predict_proba(X_test)[
                        :, classes.index("yes")
                    ]
                else:
                    scores = model.decision_function(X_test)

                    if classes.index("yes") == 0:
                        scores = -scores

                # Calculate evaluation metrics
                accuracy = accuracy_score(y_test, predictions)
                auc = roc_auc_score(y_test, scores)
                precision = precision_score(
                    y_test, predictions,
                    pos_label="yes", zero_division=0
                )
                recall = recall_score(
                    y_test, predictions,
                    pos_label="yes", zero_division=0
                )
                f1 = f1_score(
                    y_test, predictions,
                    pos_label="yes", zero_division=0
                )
                mcc = matthews_corrcoef(y_test, predictions)

                st.subheader("Model Performance")

                col1, col2, col3 = st.columns(3)

                col1.metric("Accuracy", f"{accuracy:.4f}")
                col2.metric("AUC", f"{auc:.4f}")
                col3.metric("Precision", f"{precision:.4f}")

                col4, col5, col6 = st.columns(3)

                col4.metric("Recall", f"{recall:.4f}")
                col5.metric("F1 Score", f"{f1:.4f}")
                col6.metric("MCC", f"{mcc:.4f}")

                st.subheader("Confusion Matrix")

                cm = confusion_matrix(
                    y_test,
                    predictions,
                    labels=["no", "yes"]
                )

                st.dataframe(
                    pd.DataFrame(
                        cm,
                        index=["Actual No", "Actual Yes"],
                        columns=["Predicted No", "Predicted Yes"]
                    )
                )

                st.subheader("Classification Report")

                st.text(
                    classification_report(
                        y_test,
                        predictions,
                        labels=["no", "yes"],
                        zero_division=0
                    )
                )

                st.success("Evaluation completed successfully!")


Writing app.py


In [10]:
import os

print("App file exists:", os.path.exists("app.py"))
print("App file size:", os.path.getsize("app.py"), "bytes")

print("\nStreamlit version:")
import streamlit
print(streamlit.__version__)

print("\nModel files:")
for file in os.listdir("model"):
    print(file)

App file exists: True
App file size: 4599 bytes

Streamlit version:
1.67.0

Model files:
logistic_regression.joblib
decision_tree.joblib
knn.joblib
naive_bayes.joblib
support_vector_machine.joblib


In [12]:
import socket

hostname = socket.gethostname()

print("Lab hostname:", hostname)

with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
    result = s.connect_ex(("127.0.0.1", 8501))

print("Port 8501 status:", "Already in use" if result == 0 else "Available")

Lab hostname: ip-10-3-26-253.ap-south-1.compute.internal
Port 8501 status: Available


In [14]:
import subprocess
import sys
import os

log_file = open("streamlit.log", "w")

streamlit_process = subprocess.Popen(
    [
        sys.executable, "-m", "streamlit", "run", "app.py",
        "--server.address=0.0.0.0",
        "--server.port=8501",
        "--server.headless=true",
        "--server.enableCORS=false",
        "--server.enableXsrfProtection=false"
    ],
    stdout=log_file,
    stderr=subprocess.STDOUT
)

print("Streamlit started.")
print("Process ID:", streamlit_process.pid)

Streamlit started.
Process ID: 5798


In [16]:
import time

time.sleep(5)

print("Process status:", streamlit_process.poll())

with open("streamlit.log", "r") as f:
    print(f.read())

Process status: None


2026-11-08 00:56:32.099 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://10.3.26.253:8501
  External URL: http://15.252.141.148:8501




In [19]:
import importlib.util
import requests

print("Jupyter Server Proxy installed:",
      importlib.util.find_spec("jupyter_server_proxy") is not None)

try:
    response = requests.get("http://127.0.0.1:8501", timeout=5)
    print("Streamlit status:", response.status_code)
    print("Streamlit is responding:", response.status_code == 200)
except Exception as e:
    print("Streamlit connection error:", e)

Jupyter Server Proxy installed: False
Streamlit status: 200
Streamlit is responding: True


In [21]:
%pip install jupyter-server-proxy


Note: you may need to restart the kernel to use updated packages.


In [2]:
import importlib.util

print(
    "Jupyter Server Proxy installed:",
    importlib.util.find_spec("jupyter_server_proxy") is not None
)

Jupyter Server Proxy installed: True


In [2]:
import os

print("Notebook directory:", os.getcwd())
print("App exists:", os.path.exists("app.py"))
print("Models exist:", os.path.exists("model"))
print("Test data exists:", os.path.exists("bank_marketing_test_data.csv"))

Notebook directory: /home/s2026aj05527
App exists: True
Models exist: True
Test data exists: False


In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("bank-additional-full.csv", sep=";")

X = df.drop(columns=["y", "duration"])
y = df["y"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

bank_test_data = X_test.copy()
bank_test_data["y"] = y_test

bank_test_data.to_csv(
    "bank_marketing_test_data.csv",
    index=False
)

print("Test dataset created:", bank_test_data.shape)
print("Saved successfully!")

Test dataset created: (8238, 20)
Saved successfully!


In [6]:
import subprocess
import sys

streamlit_process = subprocess.Popen(
    [
        sys.executable, "-m", "streamlit", "run", "app.py",
        "--server.address=127.0.0.1",
        "--server.port=8501",
        "--server.headless=true"
    ],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

print("Streamlit process ID:", streamlit_process.pid)

Streamlit process ID: 6375


In [8]:
import requests
import time

time.sleep(5)

try:
    response = requests.get("http://127.0.0.1:8501", timeout=10)
    print("HTTP Status:", response.status_code)
    print("Application is running:", response.status_code == 200)
except Exception as e:
    print("Error:", e)

HTTP Status: 200
Application is running: True


In [10]:
import requests
import subprocess
import sys

# Check Streamlit
try:
    r = requests.get("http://127.0.0.1:8501", timeout=5)
    print("Streamlit:", r.status_code)
except Exception as e:
    print("Streamlit error:", e)

# Check Jupyter proxy route
try:
    r = requests.get("http://127.0.0.1:8890/proxy/8501/", timeout=5)
    print("Proxy:", r.status_code)
    print("Proxy response:", r.text[:300])
except Exception as e:
    print("Proxy error:", e)

# Check whether the Jupyter server extension is enabled
result = subprocess.run(
    [sys.executable, "-m", "jupyter", "server", "extension", "list"],
    capture_output=True,
    text=True
)

print("\nJupyter extensions:")
print(result.stdout)
print(result.stderr)

Streamlit: 200
Proxy: 404
Proxy response: <!DOCTYPE HTML>
<html>

<head>

    <meta charset="utf-8">

    <title>Jupyter Server</title>
    <link id="favicon" rel="shortcut icon" type="image/x-icon" href="/static/favicon.ico?v=50afa725b5de8b00030139d09b38620224d4e7dba47c07ef0e86d4643f30c9bfe6bb7e1a4a1c561aa32834480909a4b6fe7cd1e17f7159330b6

Jupyter extensions:

Config dir: /home/s2026aj05527/.jupyter

Config dir: /home/cloud/anaconda3/etc/jupyter
    aext_assistant enabled
    - Validating aext_assistant...
Extension package aext_assistant took 0.9575s to import
      aext_assistant 4.1.0 OK
    aext_core enabled
    - Validating aext_core...
      aext_core 4.1.0 OK
    aext_panels enabled
    - Validating aext_panels...
****************** ENVIRONMENT [Panels] Environment.PRODUCTION ******************
****************** ENVIRONMENT [Panels] Environment.PRODUCTION ******************
      aext_panels 4.1.0 OK
    aext_share_notebook enabled
    - Validating aext_share_notebook...
    

In [12]:
import requests

urls = [
    "http://127.0.0.1:8890/proxy/8501",
    "http://127.0.0.1:8890/proxy/8501/",
    "http://127.0.0.1:8890/api/proxy/8501/"
]

for url in urls:
    try:
        response = requests.get(url, timeout=5)
        print(url)
        print("Status:", response.status_code)
        print("Content type:", response.headers.get("Content-Type"))
        print("---")
    except Exception as e:
        print(url, e)

http://127.0.0.1:8890/proxy/8501
Status: 404
Content type: text/html
---
http://127.0.0.1:8890/proxy/8501/
Status: 404
Content type: text/html
---
http://127.0.0.1:8890/api/proxy/8501/
Status: 404
Content type: text/html
---


In [14]:
import subprocess
import sys

result = subprocess.run(
    [
        sys.executable, "-m", "jupyter",
        "server", "extension", "list"
    ],
    capture_output=True,
    text=True
)

print(result.stdout)

In [16]:
import sys
print(sys.executable)

/home/cloud/anaconda3/bin/python


In [18]:
import subprocess
import sys

result = subprocess.run(
    [
        sys.executable,
        "-m",
        "jupyter",
        "server",
        "extension",
        "list"
    ],
    capture_output=True,
    text=True
)

print("OUTPUT:")
print(result.stdout)

print("ERRORS:")
print(result.stderr)

OUTPUT:

ERRORS:
Config dir: /home/s2026aj05527/.jupyter

Config dir: /home/cloud/anaconda3/etc/jupyter
    aext_assistant enabled
    - Validating aext_assistant...
Extension package aext_assistant took 1.0494s to import
      aext_assistant 4.1.0 OK
    aext_core enabled
    - Validating aext_core...
      aext_core 4.1.0 OK
    aext_panels enabled
    - Validating aext_panels...
****************** ENVIRONMENT [Panels] Environment.PRODUCTION ******************
****************** ENVIRONMENT [Panels] Environment.PRODUCTION ******************
      aext_panels 4.1.0 OK
    aext_share_notebook enabled
    - Validating aext_share_notebook...
      aext_share_notebook 4.1.0 OK
    aext_toolbox enabled
    - Validating aext_toolbox...
--> Alembic Config: {'db_conn_url': 'sqlite:////home/s2026aj05527/anaconda_projects/db/project_filebrowser.db', 'db_path': '/home/s2026aj05527/anaconda_projects/db/project_filebrowser.db'}
Default dir: /home/s2026aj05527. Running chdir...
Checking for databas

In [20]:
import requests

response = requests.get("http://127.0.0.1:8501", timeout=10)

print("Status:", response.status_code)
print("Streamlit page loaded:", "Bank Marketing Classification" in response.text)
print("Page response length:", len(response.text))

Status: 200
Streamlit page loaded: False
Page response length: 7863


In [22]:
import pandas as pd

test_df = pd.read_csv("bank_marketing_test_data.csv")

print("Dataset shape:", test_df.shape)
print("\nColumns:", test_df.columns.tolist())
print("\nTarget distribution:")
print(test_df["y"].value_counts())

Dataset shape: (8238, 20)

Columns: ['age', 'job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'day_of_week', 'campaign', 'pdays', 'previous', 'poutcome', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed', 'y']

Target distribution:
y
no     7310
yes     928
Name: count, dtype: int64


In [24]:
import joblib
import os

model_files = sorted(os.listdir("model"))

for filename in model_files:
    model = joblib.load(os.path.join("model", filename))
    print(filename, "— Loaded successfully")

print("\nTotal models:", len(model_files))

decision_tree.joblib — Loaded successfully
knn.joblib — Loaded successfully
logistic_regression.joblib — Loaded successfully
naive_bayes.joblib — Loaded successfully
support_vector_machine.joblib — Loaded successfully

Total models: 5


In [26]:
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef
)

X_test = test_df.drop(columns=["y"])
y_test = test_df["y"]

for filename in sorted(os.listdir("model")):

    model = joblib.load(os.path.join("model", filename))
    predictions = model.predict(X_test)

    classifier = model.named_steps["classifier"]
    classes = list(classifier.classes_)

    if hasattr(model, "predict_proba"):
        scores = model.predict_proba(X_test)[:, classes.index("yes")]
    else:
        scores = model.decision_function(X_test)
        if classes.index("yes") == 0:
            scores = -scores

    print("\nModel:", filename)
    print("Accuracy:", round(accuracy_score(y_test, predictions), 4))
    print("AUC:", round(roc_auc_score(y_test, scores), 4))
    print("Precision:", round(precision_score(
        y_test, predictions, pos_label="yes"
    ), 4))
    print("Recall:", round(recall_score(
        y_test, predictions, pos_label="yes"
    ), 4))
    print("F1 Score:", round(f1_score(
        y_test, predictions, pos_label="yes"
    ), 4))
    print("MCC:", round(matthews_corrcoef(y_test, predictions), 4))


Model: decision_tree.joblib
Accuracy: 0.8418
AUC: 0.6258
Precision: 0.3131
Recall: 0.3384
F1 Score: 0.3252
MCC: 0.236

Model: knn.joblib
Accuracy: 0.8972
AUC: 0.7444
Precision: 0.5839
Recall: 0.3039
F1 Score: 0.3997
MCC: 0.3719

Model: logistic_regression.joblib
Accuracy: 0.9009
AUC: 0.8008
Precision: 0.6905
Recall: 0.2188
F1 Score: 0.3322
MCC: 0.3516

Model: naive_bayes.joblib
Accuracy: 0.8049
AUC: 0.7755
Precision: 0.3172
Recall: 0.6347
F1 Score: 0.423
MCC: 0.349

Model: support_vector_machine.joblib
Accuracy: 0.9014
AUC: 0.7979
Precision: 0.7164
Recall: 0.2069
F1 Score: 0.3211
MCC: 0.3502


In [28]:
import streamlit
import os
import pandas as pd

print("APP VALIDATION")
print("------------------------")

print("app.py exists:", os.path.exists("app.py"))
print("Test CSV exists:", os.path.exists("bank_marketing_test_data.csv"))
print("Model count:", len(os.listdir("model")))

print("Streamlit version:", streamlit.__version__)

print("\nAll project components are available.")

APP VALIDATION
------------------------
app.py exists: True
Test CSV exists: True
Model count: 5
Streamlit version: 1.67.0

All project components are available.


In [30]:
%%writefile requirements.txt
streamlit==1.67.0
pandas==2.2.3
numpy==2.5.4
scikit-learn==1.9.1
joblib

Writing requirements.txt


In [32]:
import subprocess
import sys

commands = [
    ["jupyter", "server", "list"],
    ["jupyter", "notebook", "--version"],
    ["jupyter", "server", "--version"]
]

for command in commands:
    print("\nCOMMAND:", " ".join(command))
    result = subprocess.run(
        command,
        capture_output=True,
        text=True
    )
    print(result.stdout)
    print(result.stderr)


COMMAND: jupyter server list
Currently running servers:
http://localhost:8888/?token=732426ed82adbf200634373922e278d917cd674f421a07dc :: /home/s2026aj05527
http://localhost:8889/?token=166ea1d9a110e19c92c03bdd7b2e16130391f53a1754f67c :: /home/s2026aj05527
http://localhost:8889/?token=5d67d4580ce4bba3f4025366ad59fcf78cdfcaeaf2b711b0 :: /home/s2026aj05527
http://localhost:8890/?token=f2bff562c3ba4dc44a8c876797b705213dce87a751080a02 :: /home/s2026aj05527



COMMAND: jupyter notebook --version
7.2.2



COMMAND: jupyter server --version
2.14.1




In [34]:
import requests

ports = [8888, 8889, 8890]

for port in ports:
    print(f"\nChecking Jupyter port: {port}")

    for path in ["/api/status", "/proxy/8501/"]:
        try:
            response = requests.get(
                f"http://127.0.0.1:{port}{path}",
                timeout=5
            )
            print(path, "->", response.status_code)
        except Exception as e:
            print(path, "->", str(e))


Checking Jupyter port: 8888
/api/status -> 403
/proxy/8501/ -> 404

Checking Jupyter port: 8889
/api/status -> 403
/proxy/8501/ -> 404

Checking Jupyter port: 8890
/api/status -> 403
/proxy/8501/ -> 404
